In [13]:
# Cell 1: Install all required packages
!pip install rank-bm25 sentence-transformers transformers torch google-generativeai groq

In [14]:
# Cell 2: Imports and API Configuration
import os
import numpy as np
import google.generativeai as genai
from getpass import getpass
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity

# This will create a popup box for you to paste your keys
GEMINI_API_KEY = getpass("Enter Gemini API Key: ")
GROQ_API_KEY   = getpass("Enter Groq API Key: ")

if GEMINI_API_KEY:
    genai.configure(api_key=GEMINI_API_KEY)

print("Setup complete")

Enter Gemini API Key: ··········
Enter Groq API Key: ··········
Setup complete


In [15]:
# Cell 3: Part 1 - Document Corpus (10+ AI/ML documents)

corpus = [
    # Transformers / Attention (3 related but distinct)
    "The attention mechanism allows a transformer to weigh the importance of each token in a sequence when encoding another token, capturing long-range dependencies.",
    "Multi-head attention splits the embedding into multiple heads, each learning different types of relationships such as syntactic or semantic patterns simultaneously.",
    "Self-attention computes queries, keys, and values from the same input sequence, enabling the model to relate each word to every other word in the sentence.",

    # Neural Network Training (3 related but distinct)
    "Gradient descent updates model parameters by computing the gradient of the loss function and stepping in the direction that reduces the loss.",
    "Adam optimizer combines momentum and adaptive learning rates, making it robust to sparse gradients and suitable for training deep neural networks.",
    "Batch normalization normalizes activations within a mini-batch to stabilize and accelerate training by reducing internal covariate shift.",

    # Embeddings & Representations
    "Word2Vec uses a shallow neural network to learn dense vector representations of words by predicting surrounding context words in a sliding window.",
    "Sentence-BERT (SBERT) fine-tunes BERT using siamese networks and a contrastive objective to produce semantically meaningful sentence embeddings.",

    # Retrieval & RAG
    "BM25 is a probabilistic keyword-based retrieval algorithm that ranks documents by term frequency and inverse document frequency, excelling at exact-match queries.",
    "Retrieval-Augmented Generation (RAG) grounds large language model outputs in external documents, reducing hallucinations by injecting retrieved context into the prompt.",

    # Miscellaneous technical jargon doc (good for BM25)
    "LoRA (Low-Rank Adaptation) inserts trainable low-rank matrices into frozen pre-trained transformer layers, enabling parameter-efficient fine-tuning at a fraction of the cost.",
]

print(f"Corpus size: {len(corpus)} documents")
for i, doc in enumerate(corpus):
    print(f"  [{i:02d}] {doc[:80]}...")

Corpus size: 11 documents
  [00] The attention mechanism allows a transformer to weigh the importance of each tok...
  [01] Multi-head attention splits the embedding into multiple heads, each learning dif...
  [02] Self-attention computes queries, keys, and values from the same input sequence, ...
  [03] Gradient descent updates model parameters by computing the gradient of the loss ...
  [04] Adam optimizer combines momentum and adaptive learning rates, making it robust t...
  [05] Batch normalization normalizes activations within a mini-batch to stabilize and ...
  [06] Word2Vec uses a shallow neural network to learn dense vector representations of ...
  [07] Sentence-BERT (SBERT) fine-tunes BERT using siamese networks and a contrastive o...
  [08] BM25 is a probabilistic keyword-based retrieval algorithm that ranks documents b...
  [09] Retrieval-Augmented Generation (RAG) grounds large language model outputs in ext...
  [10] LoRA (Low-Rank Adaptation) inserts trainable low-rank mat

In [16]:
# Cell 4: Part 2 - HybridRetriever

class HybridRetriever:
    def __init__(self, corpus: list[str], k: int = 60):
        """
        k: RRF constant (default 60, as per the original RRF paper).
        """
        self.corpus = corpus
        self.k = k

        # --- BM25 index ---
        tokenized = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(tokenized)

        # --- SBERT index ---
        print("Loading SBERT model...")
        self.sbert = SentenceTransformer("all-MiniLM-L6-v2")
        self.corpus_embeddings = self.sbert.encode(corpus, convert_to_numpy=True)
        print("HybridRetriever ready ✓")

    def retrieve(self, query: str, top_k: int = 5) -> list[dict]:
        """
        Returns top_k documents fused via Reciprocal Rank Fusion (RRF).
        Each result: {"doc_id", "rrf_score", "bm25_rank", "sbert_rank", "text"}
        """
        n = len(self.corpus)

        # --- BM25 ranking ---
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_ranked = np.argsort(bm25_scores)[::-1]           # descending
        bm25_rank = {doc_id: rank + 1 for rank, doc_id in enumerate(bm25_ranked)}

        # --- SBERT ranking ---
        query_emb = self.sbert.encode([query], convert_to_numpy=True)
        sbert_scores = cosine_similarity(query_emb, self.corpus_embeddings)[0]
        sbert_ranked = np.argsort(sbert_scores)[::-1]          # descending
        sbert_rank = {doc_id: rank + 1 for rank, doc_id in enumerate(sbert_ranked)}

        # --- RRF fusion ---
        rrf_scores = {}
        for doc_id in range(n):
            rrf_scores[doc_id] = (
                1 / (self.k + bm25_rank[doc_id]) +
                1 / (self.k + sbert_rank[doc_id])
            )

        # Sort by RRF score descending
        sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

        results = []
        for doc_id, rrf_score in sorted_docs[:top_k]:
            results.append({
                "doc_id":     doc_id,
                "rrf_score":  round(rrf_score, 6),
                "bm25_rank":  bm25_rank[doc_id],
                "sbert_rank": sbert_rank[doc_id],
                "text":       self.corpus[doc_id],
            })
        return results


# Initialise once — reused across all parts
retriever = HybridRetriever(corpus)

# Quick smoke test
test_results = retriever.retrieve("how does attention work?", top_k=3)
print("\nSmoke test — 'how does attention work?'")
for r in test_results:
    print(f"  RRF={r['rrf_score']}  BM25_rank={r['bm25_rank']}  SBERT_rank={r['sbert_rank']}")
    print(f"    → {r['text'][:90]}")

Loading SBERT model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HybridRetriever ready ✓

Smoke test — 'how does attention work?'
  RRF=0.032266  BM25_rank=1  SBERT_rank=3
    → Multi-head attention splits the embedding into multiple heads, each learning different typ
  RRF=0.032258  BM25_rank=2  SBERT_rank=2
    → The attention mechanism allows a transformer to weigh the importance of each token in a se
  RRF=0.030769  BM25_rank=5  SBERT_rank=5
    → Retrieval-Augmented Generation (RAG) grounds large language model outputs in external docu


In [17]:
# Cell 5: Part 3 - Cross-Encoder Re-Ranker

print("Loading cross-encoder model...")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Cross-encoder ready")

def rerank(query: str, candidates: list[dict], top_k: int = 3) -> list[dict]:
    """
    Re-ranks candidate documents using a cross-encoder.

    Args:
        query:      The ORIGINAL user query (not the HyDE-expanded version).
        candidates: List of dicts from HybridRetriever.retrieve()
        top_k:      Number of top documents to return after re-ranking.

    Returns:
        List of dicts with an extra "ce_score" key, sorted by cross-encoder score.
    """
    # Build (query, passage) pairs for the cross-encoder
    pairs = [(query, c["text"]) for c in candidates]
    ce_scores = cross_encoder.predict(pairs)   # scores can be negative; higher = more relevant

    # Attach scores and sort
    for candidate, score in zip(candidates, ce_scores):
        candidate["ce_score"] = round(float(score), 4)

    reranked = sorted(candidates, key=lambda x: x["ce_score"], reverse=True)
    return reranked[:top_k]


# Smoke test
candidates = retriever.retrieve("attention mechanism in transformers", top_k=5)
reranked   = rerank("attention mechanism in transformers", candidates, top_k=3)

print("\nSmoke test — re-ranked results:")
for r in reranked:
    print(f"  CE_score={r['ce_score']:>8.4f}  → {r['text'][:90]}")

Loading cross-encoder model...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cross-encoder ready

Smoke test — re-ranked results:
  CE_score=  7.3007  → The attention mechanism allows a transformer to weigh the importance of each token in a se
  CE_score= -6.8492  → Self-attention computes queries, keys, and values from the same input sequence, enabling t
  CE_score= -6.9310  → Multi-head attention splits the embedding into multiple heads, each learning different typ


In [18]:
# Cell 6: Part 4 - Query Expansion via HyDE using Gemini

gemini_model = genai.GenerativeModel("gemini-2.5-flash")

def hyde_expand(user_query: str) -> str:
    """
    HyDE: ask Gemini to write a short hypothetical answer to the query.
    That hypothetical document becomes the retrieval query instead.
    Uses temperature=0.0 for deterministic output.
    """
    prompt = (
        "You are an expert in AI and machine learning. "
        "Write a concise, factual 2-3 sentence answer to the following question "
        "as if it appeared in a textbook. Do not add introductions or caveats.\n\n"
        f"Question: {user_query}"
    )
    response = gemini_model.generate_content(
        prompt,
        generation_config=genai.GenerationConfig(temperature=0.0, max_output_tokens=150),
    )
    return response.text.strip()


# Smoke test
sample_query = "how do transformers encode meaning?"
hyde_doc = hyde_expand(sample_query)
print(f"Original query : {sample_query}")
print(f"HyDE expansion : {hyde_doc}")

Original query : how do transformers encode meaning?
HyDE expansion : Transformers encode meaning primarily through self


In [19]:
# Cell 7: Part 5 - End-to-End Advanced RAG Pipeline

from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY)

def naive_rag(user_query: str, top_k: int = 3) -> str:
    """
    Baseline: Dense-only retrieval (SBERT cosine), no expansion, no re-ranking.
    """
    query_emb     = retriever.sbert.encode([user_query], convert_to_numpy=True)
    sbert_scores  = cosine_similarity(query_emb, retriever.corpus_embeddings)[0]
    top_indices   = np.argsort(sbert_scores)[::-1][:top_k]
    top_docs      = [corpus[i] for i in top_indices]

    context = "\n".join(f"- {d}" for d in top_docs)
    prompt  = (
        f"Answer the question using ONLY the context below.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {user_query}\nAnswer:"
    )
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=300,
    )
    return response.choices[0].message.content.strip(), top_docs


def advanced_rag(user_query: str) -> str:
    """
    Full pipeline:
        1. Query Expansion (HyDE via Gemini)
        2. Hybrid Retrieval (BM25 + SBERT + RRF)
        3. Cross-Encoder Re-Ranking
        4. LLM Generation (Groq / LLaMA-3)

    Returns the final answer string.
    """
    # Step 1 — HyDE expansion
    expanded_query = hyde_expand(user_query)
    print(f"  [HyDE] {expanded_query[:100]}...")

    # Step 2 — Hybrid retrieval using the expanded query
    candidates = retriever.retrieve(expanded_query, top_k=6)

    # Step 3 — Re-rank using the ORIGINAL user query (not the expanded one)
    top_docs = rerank(user_query, candidates, top_k=3)

    # Step 4 — Generate answer
    context = "\n".join(f"- {d['text']}" for d in top_docs)
    prompt  = (
        f"Answer the question using ONLY the context below.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {user_query}\nAnswer:"
    )
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=300,
    )
    answer = response.choices[0].message.content.strip()
    return answer, top_docs


# Quick end-to-end test
q = "how do transformers encode meaning?"
print(f"\nRunning Advanced RAG on: '{q}'\n")
answer, top_docs = advanced_rag(q)
print("\nTop docs used:")
for d in top_docs:
    print(f"  CE={d['ce_score']:>7.3f} → {d['text'][:80]}")
print(f"\nAnswer:\n{answer}")


Running Advanced RAG on: 'how do transformers encode meaning?'

  [HyDE] Transformers encode meaning primarily through self...

Top docs used:
  CE= -6.481 → The attention mechanism allows a transformer to weigh the importance of each tok
  CE=-10.370 → LoRA (Low-Rank Adaptation) inserts trainable low-rank matrices into frozen pre-t
  CE=-11.196 → Self-attention computes queries, keys, and values from the same input sequence, 

Answer:
Transformers encode meaning by weighing the importance of each token in a sequence through the attention mechanism, which captures long-range dependencies. Specifically, self-attention allows the model to relate each word to every other word in the sentence by computing queries, keys, and values from the same input sequence.


In [20]:
# Cell 8: Part 6 - Comparison Experiment (Naïve RAG vs Advanced RAG)

test_queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what is parameter efficient fine tuning?",   # your own query
]

results_table = []

for query in test_queries:
    print("=" * 70)
    print(f"Query: {query}")

    # Naïve RAG
    naive_answer, naive_top = naive_rag(query, top_k=3)
    naive_top_doc = naive_top[0]

    # Advanced RAG
    adv_answer, adv_top = advanced_rag(query)
    adv_top_doc = adv_top[0]["text"]

    different = " Yes" if naive_top_doc != adv_top_doc else " No"

    results_table.append({
        "Query":                   query,
        "Naïve RAG Top Doc":       naive_top_doc[:80] + "...",
        "Advanced RAG Top Doc":    adv_top_doc[:80]   + "...",
        "Are they different?":     different,
        "Naïve Answer":            naive_answer,
        "Advanced Answer":         adv_answer,
    })

    print(f"\n  Naïve    top doc  : {naive_top_doc[:90]}")
    print(f"  Advanced top doc  : {adv_top_doc[:90]}")
    print(f"  Different?        : {different}")
    print(f"\n  Naïve Answer    → {naive_answer[:200]}")
    print(f"\n  Advanced Answer → {adv_answer[:200]}")
    print()

Query: how do transformers encode meaning?
  [HyDE] Transformers encode meaning primarily through self...

  Naïve    top doc  : The attention mechanism allows a transformer to weigh the importance of each token in a se
  Advanced top doc  : The attention mechanism allows a transformer to weigh the importance of each token in a se
  Different?        :  No

  Naïve Answer    → Transformers encode meaning by weighing the importance of each token in a sequence when encoding another token, capturing long-range dependencies through the attention mechanism.

  Advanced Answer → Transformers encode meaning by weighing the importance of each token in a sequence when encoding another token, capturing long-range dependencies through the attention mechanism, and relating each wor

Query: optimization techniques for training
  [HyDE] Optimization techniques are algorithms used during...

  Naïve    top doc  : Gradient descent updates model parameters by computing the gradient of the loss function

In [21]:
# Cell 9: Pretty-print the comparison table

header = "| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Are they different? |"
sep    = "|---|---|---|---|"
rows   = []

for r in results_table:
    row = f"| {r['Query']} | {r['Naïve RAG Top Doc']} | {r['Advanced RAG Top Doc']} | {r['Are they different?']} |"
    rows.append(row)

table = "\n".join([header, sep] + rows)
print(table)

| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Are they different? |
|---|---|---|---|
| how do transformers encode meaning? | The attention mechanism allows a transformer to weigh the importance of each tok... | The attention mechanism allows a transformer to weigh the importance of each tok... |  No |
| optimization techniques for training | Gradient descent updates model parameters by computing the gradient of the loss ... | Adam optimizer combines momentum and adaptive learning rates, making it robust t... |  Yes |
| what is parameter efficient fine tuning? | LoRA (Low-Rank Adaptation) inserts trainable low-rank matrices into frozen pre-t... | LoRA (Low-Rank Adaptation) inserts trainable low-rank matrices into frozen pre-t... |  No |


In [22]:
# Cell 10 (Bonus 1) - Weighted RRF with alpha tuning

def weighted_rrf_retrieve(query: str, alpha: float = 0.5, top_k: int = 5) -> list[dict]:
    """
    RRF_weighted(d) = alpha * 1/(k + r_BM25) + (1-alpha) * 1/(k + r_SBERT)
    """
    k = retriever.k
    n = len(corpus)

    bm25_scores  = retriever.bm25.get_scores(query.lower().split())
    bm25_ranked  = np.argsort(bm25_scores)[::-1]
    bm25_rank    = {doc_id: rank + 1 for rank, doc_id in enumerate(bm25_ranked)}

    query_emb    = retriever.sbert.encode([query], convert_to_numpy=True)
    sbert_scores = cosine_similarity(query_emb, retriever.corpus_embeddings)[0]
    sbert_ranked = np.argsort(sbert_scores)[::-1]
    sbert_rank   = {doc_id: rank + 1 for rank, doc_id in enumerate(sbert_ranked)}

    rrf_scores = {
        doc_id: alpha * (1 / (k + bm25_rank[doc_id])) +
                (1 - alpha) * (1 / (k + sbert_rank[doc_id]))
        for doc_id in range(n)
    }

    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return [
        {"doc_id": doc_id, "weighted_rrf": round(score, 6),
         "bm25_rank": bm25_rank[doc_id], "sbert_rank": sbert_rank[doc_id],
         "text": corpus[doc_id]}
        for doc_id, score in sorted_docs[:top_k]
    ]


# Experiment: keyword-heavy query vs semantic query across alpha values
queries_bonus = {
    "keyword-heavy": "BM25 term frequency inverse document frequency",
    "semantic":      "how do models understand the meaning of sentences",
}

for label, q in queries_bonus.items():
    print(f"\n--- {label}: '{q}' ---")
    for alpha in [0.3, 0.5, 0.7]:
        top = weighted_rrf_retrieve(q, alpha=alpha, top_k=1)[0]
        print(f"  alpha={alpha}  top doc → {top['text'][:80]}")


--- keyword-heavy: 'BM25 term frequency inverse document frequency' ---
  alpha=0.3  top doc → BM25 is a probabilistic keyword-based retrieval algorithm that ranks documents b
  alpha=0.5  top doc → BM25 is a probabilistic keyword-based retrieval algorithm that ranks documents b
  alpha=0.7  top doc → BM25 is a probabilistic keyword-based retrieval algorithm that ranks documents b

--- semantic: 'how do models understand the meaning of sentences' ---
  alpha=0.3  top doc → Self-attention computes queries, keys, and values from the same input sequence, 
  alpha=0.5  top doc → Self-attention computes queries, keys, and values from the same input sequence, 
  alpha=0.7  top doc → Self-attention computes queries, keys, and values from the same input sequence, 


In [23]:
# Cell 11 (Bonus 2) - Chunk Size Study

# A longer document (>500 words) about Transformers
long_document = """
The Transformer architecture, introduced in the seminal paper "Attention Is All You Need" by Vaswani et al. in 2017, revolutionized the field of natural language processing and has since become the dominant paradigm for a wide range of sequence-to-sequence tasks. Prior to the Transformer, recurrent neural networks (RNNs) and long short-term memory networks (LSTMs) were the go-to architectures for modeling sequential data. However, these models suffered from significant limitations: they processed tokens sequentially, making parallelization during training difficult, and they struggled to capture long-range dependencies due to vanishing gradients over long sequences.

The core innovation of the Transformer is the self-attention mechanism, which allows the model to directly relate each token in a sequence to every other token, regardless of their distance. This is achieved through the computation of queries, keys, and values from the input embeddings. The dot product of queries and keys is scaled and passed through a softmax to produce attention weights, which are then used to compute a weighted sum of the values. By doing this for all tokens simultaneously, the Transformer can capture complex dependencies across the full sequence in a single step.

Multi-head attention extends this concept by running multiple self-attention operations in parallel, each with its own learned projection matrices. Each attention head can focus on different aspects of the input — for example, one head might capture syntactic relationships while another captures semantic similarities. The outputs of all heads are concatenated and projected back to the model's hidden dimension, giving the network a rich, multi-perspective representation of the input.

The Transformer also incorporates positional encodings to inject information about the order of tokens, since self-attention itself is permutation-invariant. These encodings are typically fixed sinusoidal functions of the token position and are added directly to the input embeddings before being fed into the first attention layer.

Feed-forward sub-layers are interleaved with the attention layers in each Transformer block. These consist of two linear projections with a ReLU activation in between, applied independently to each token position. Layer normalization and residual connections around each sub-layer help stabilize training and allow gradients to flow smoothly through the deep network.

The encoder processes the input sequence into a sequence of context-aware representations, while the decoder generates the output sequence one token at a time. During decoding, masked self-attention ensures that each output token can only attend to previously generated tokens, preserving the autoregressive property. Cross-attention layers in the decoder allow each output token to attend to the full encoder output, enabling the decoder to focus on the most relevant parts of the input for each generation step.

Training Transformers at scale requires careful optimization strategies. The Adam optimizer with a warmup learning rate schedule is standard practice, as it helps the model converge quickly without overshooting at the start. Dropout is applied to attention weights and feed-forward activations as a regularization technique. For very large models, gradient clipping is used to prevent exploding gradients.

The Transformer has proven highly scalable: increasing the number of layers, attention heads, and hidden dimensions generally improves performance, following a predictable power law as a function of compute. This scalability has driven the development of large language models such as GPT, BERT, T5, and PaLM, each pushing the boundaries of what is achievable with pre-training on large text corpora followed by fine-tuning on downstream tasks.
"""

def chunk_document(text: str, chunk_size: int) -> list[str]:
    """Split text into non-overlapping chunks of exactly chunk_size words."""
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i : i + chunk_size])
        chunks.append(chunk)
    return chunks


def build_retriever_for_chunks(chunks: list[str]):
    """Build a BM25 + SBERT retriever over the given chunk list."""
    tokenized = [c.lower().split() for c in chunks]
    bm25 = BM25Okapi(tokenized)
    embeddings = retriever.sbert.encode(chunks, convert_to_numpy=True)
    return bm25, embeddings


def retrieve_from_chunks(query: str, chunks, bm25, embeddings, top_k: int = 3):
    """Hybrid BM25+SBERT RRF retrieval over a custom chunk list."""
    k = 60
    n = len(chunks)

    bm25_scores  = bm25.get_scores(query.lower().split())
    bm25_ranked  = np.argsort(bm25_scores)[::-1]
    bm25_rank    = {i: rank + 1 for rank, i in enumerate(bm25_ranked)}

    query_emb    = retriever.sbert.encode([query], convert_to_numpy=True)
    sbert_scores = cosine_similarity(query_emb, embeddings)[0]
    sbert_ranked = np.argsort(sbert_scores)[::-1]
    sbert_rank   = {i: rank + 1 for rank, i in enumerate(sbert_ranked)}

    rrf_scores = {
        i: 1 / (k + bm25_rank[i]) + 1 / (k + sbert_rank[i])
        for i in range(n)
    }

    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return [
        {"chunk_id": i, "rrf_score": round(s, 6),
         "bm25_rank": bm25_rank[i], "sbert_rank": sbert_rank[i],
         "text": chunks[i]}
        for i, s in sorted_docs[:top_k]
    ]


# Print word count
word_count = len(long_document.split())
print(f"Document length: {word_count} words\n")

# Queries to test
chunk_queries = [
    "how does self-attention work in transformers",
    "what optimizer is used to train transformers",
]

for chunk_size in [50, 100, 200]:
    chunks = chunk_document(long_document, chunk_size)
    bm25, embeddings = build_retriever_for_chunks(chunks)
    print(f"{'='*70}")
    print(f"Chunk size: {chunk_size} words  →  {len(chunks)} chunks total")
    print(f"{'='*70}")
    for query in chunk_queries:
        print(f"\n  Query: \"{query}\"")
        top = retrieve_from_chunks(query, chunks, bm25, embeddings, top_k=1)[0]
        print(f"  Top chunk (RRF={top['rrf_score']}):")
        print(f"  '{top['text'][:200]}...'" if len(top['text']) > 200 else f"  '{top['text']}'")
    print()

Document length: 554 words

Chunk size: 50 words  →  12 chunks total

  Query: "how does self-attention work in transformers"
  Top chunk (RRF=0.032258):
  'softmax to produce attention weights, which are then used to compute a weighted sum of the values. By doing this for all tokens simultaneously, the Transformer can capture complex dependencies across ...'

  Query: "what optimizer is used to train transformers"
  Top chunk (RRF=0.032787):
  'in the decoder allow each output token to attend to the full encoder output, enabling the decoder to focus on the most relevant parts of the input for each generation step. Training Transformers at sc...'

Chunk size: 100 words  →  6 chunks total

  Query: "how does self-attention work in transformers"
  Top chunk (RRF=0.032018):
  'in the decoder allow each output token to attend to the full encoder output, enabling the decoder to focus on the most relevant parts of the input for each generation step. Training Transformers at sc...'

  Query: 

In [24]:
# Cell 12 (Bonus 3) - ColBERT MaxSim as Third Retriever + Three-way RRF Fusion

def colbert_maxsim_scores(query: str, corpus_texts: list[str]) -> np.ndarray:
    """
    Approximate ColBERT MaxSim scoring using SBERT token-level embeddings.

    For each document:
      - Encode each query word individually  →  Q  (q_len x D)
      - Encode each document word individually  →  D  (d_len x D)
      - MaxSim = sum over query tokens of max cosine similarity with any doc token
    """
    query_tokens = query.split()
    if not query_tokens:
        return np.zeros(len(corpus_texts))

    Q = retriever.sbert.encode(query_tokens, convert_to_numpy=True, show_progress_bar=False)
    Q = Q / (np.linalg.norm(Q, axis=1, keepdims=True) + 1e-9)

    scores = []
    for doc_text in corpus_texts:
        doc_tokens = doc_text.split()
        if not doc_tokens:
            scores.append(0.0)
            continue
        D_mat = retriever.sbert.encode(doc_tokens, convert_to_numpy=True, show_progress_bar=False)
        D_mat = D_mat / (np.linalg.norm(D_mat, axis=1, keepdims=True) + 1e-9)
        sim_matrix = Q @ D_mat.T
        maxsim = sim_matrix.max(axis=1).sum()
        scores.append(float(maxsim))

    return np.array(scores)


def three_way_rrf_retrieve(query: str, top_k: int = 5) -> list[dict]:
    """
    Fuse three ranked lists — BM25, SBERT, ColBERT MaxSim — using RRF.

    RRF(d) = 1/(k + r_BM25) + 1/(k + r_SBERT) + 1/(k + r_ColBERT)
    """
    k = retriever.k
    n = len(corpus)

    # BM25 ranking
    bm25_scores  = retriever.bm25.get_scores(query.lower().split())
    bm25_ranked  = np.argsort(bm25_scores)[::-1]
    bm25_rank    = {doc_id: rank + 1 for rank, doc_id in enumerate(bm25_ranked)}

    # SBERT ranking
    query_emb    = retriever.sbert.encode([query], convert_to_numpy=True)
    sbert_scores = cosine_similarity(query_emb, retriever.corpus_embeddings)[0]
    sbert_ranked = np.argsort(sbert_scores)[::-1]
    sbert_rank   = {doc_id: rank + 1 for rank, doc_id in enumerate(sbert_ranked)}

    # ColBERT ranking
    print("  [ColBERT] Computing MaxSim scores...", end=" ", flush=True)
    colbert_scores = colbert_maxsim_scores(query, corpus)
    colbert_ranked = np.argsort(colbert_scores)[::-1]
    colbert_rank   = {doc_id: rank + 1 for rank, doc_id in enumerate(colbert_ranked)}
    print("done")

    # Three-way RRF fusion
    rrf_scores = {
        doc_id: (
            1 / (k + bm25_rank[doc_id]) +
            1 / (k + sbert_rank[doc_id]) +
            1 / (k + colbert_rank[doc_id])
        )
        for doc_id in range(n)
    }

    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return [
        {
            "doc_id":        doc_id,
            "three_way_rrf": round(score, 6),
            "bm25_rank":     bm25_rank[doc_id],
            "sbert_rank":    sbert_rank[doc_id],
            "colbert_rank":  colbert_rank[doc_id],
            "text":          corpus[doc_id],
        }
        for doc_id, score in sorted_docs[:top_k]
    ]


# Comparison: 2-way RRF vs 3-way RRF
test_queries_b3 = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what is parameter efficient fine tuning?",
]

print("Comparing 2-way RRF (BM25+SBERT) vs 3-way RRF (BM25+SBERT+ColBERT)\n")
print("=" * 70)

for query in test_queries_b3:
    print(f"\nQuery: \"{query}\"")
    two_way_top   = retriever.retrieve(query, top_k=1)[0]
    three_way_top = three_way_rrf_retrieve(query, top_k=1)[0]
    same = " same" if two_way_top["text"] == three_way_top["text"] else " different"
    print(f"  2-way  top doc → {two_way_top['text'][:90]}")
    print(f"  3-way  top doc → {three_way_top['text'][:90]}")
    print(f"  Result changed? {same}")

print("\n" + "=" * 70)
print("\nFull top-3 for 3-way RRF on first query:")
results = three_way_rrf_retrieve(test_queries_b3[0], top_k=3)
for i, r in enumerate(results, 1):
    print(f"  {i}. [RRF={r['three_way_rrf']}] BM25={r['bm25_rank']} SBERT={r['sbert_rank']} ColBERT={r['colbert_rank']}")
    print(f"     {r['text'][:100]}")

Comparing 2-way RRF (BM25+SBERT) vs 3-way RRF (BM25+SBERT+ColBERT)


Query: "how do transformers encode meaning?"
  [ColBERT] Computing MaxSim scores... done
  2-way  top doc → LoRA (Low-Rank Adaptation) inserts trainable low-rank matrices into frozen pre-trained tra
  3-way  top doc → LoRA (Low-Rank Adaptation) inserts trainable low-rank matrices into frozen pre-trained tra
  Result changed?  same

Query: "optimization techniques for training"
  [ColBERT] Computing MaxSim scores... done
  2-way  top doc → Adam optimizer combines momentum and adaptive learning rates, making it robust to sparse g
  3-way  top doc → Adam optimizer combines momentum and adaptive learning rates, making it robust to sparse g
  Result changed?  same

Query: "what is parameter efficient fine tuning?"
  [ColBERT] Computing MaxSim scores... done
  2-way  top doc → LoRA (Low-Rank Adaptation) inserts trainable low-rank matrices into frozen pre-trained tra
  3-way  top doc → LoRA (Low-Rank Adaptation) inserts trai